# Solving the 6 Player Intransitive Dice Problem 

After I had published my 4- and 5-player problems, I was contacted by another recreational mathematician, Youhua Li, who had a solution for the four-player problem that involved fewer dice.  I discoverd that his solution used a set of dice that could be generated by an arbitrary domination tournament graph, and this "publication" intends to prove that.  

In addition, another recreational mathematician John Robert Marshall commented on the math_problems repo that he had beaten me to the punch with a published solution for four-player dice. These are notably is easy to extend to five- and six-player solutions. As he noted (in a private communication) that an isomorphic set to the 71 dice in my five-player solution was verified already, we should check those as well for the six-player variant in particular.

First, some imports and functions from the last one:

In [1]:
# First let's import
import itertools as it
from tqdm import tqdm
from joblib import delayed,Parallel
from tabulate import tabulate

We also define some previous functions that are involved in the checking.  In particular, we modify the checking function to check only values that include the minimum value and are above it.  So for instance, the subcombinations for n=7 that include 1 are:

In [2]:
def get_players(n): return sum([n>=i for i in [1, 3, 7, 19, 67, 331, 1163]])
def get_sub_comb(n,minimum): return ((minimum,*j) for j in it.combinations([i for i in range(n) if i>minimum], get_players(n)-2))
def get_win_bias(d1,d2): return sum(((j<i)-(i<j) for i,j in zip(d1,d2)))
def get_winners(dice): return tuple(tuple(d1[0] for d1 in dice if get_win_bias(d1,d2)>0) for d2 in dice)
def has_full_coverage(winners,n,minimum):
    samples = get_sub_comb(n,minimum)
    set_winners = list(map(set,winners))
    return all(any(all(i in w for i in s) for w in set_winners) for s in samples)

get_sub_comb(7,1)

<generator object get_sub_comb.<locals>.<genexpr> at 0x747fb0375f20>

In addition, we define the functions to generate the dice:

In [3]:
def li_dice(n):
    assert not (n+1)%4
    w = tuple(sorted([((i+1)**2)%n for i in range(n//2)]))
    return list(zip(*sorted([[(i+j*k)%n+i*n*len(w)+w.index(j)*n for k in range(n)] for i in range(n) for j in w])))
def marshall_dice(n):
    assert not (n+1)%8
    g = tuple([(2**i)%n for i in range(n//2)])
    return [tuple([(i*j+n//2)%n+k*n for k,i in enumerate(g)]) for j in range(-n//2+1,n//2+1)]

Let's quickly check our coverage testing function. Let's give it a positive result, testing for the tournament that represents a solution for the Oskar (n=7) dice:

In [4]:
n = 7
dice = li_dice(n)
winners = get_winners(dice)
print(winners)
print({i:has_full_coverage(winners,n,i) for i in range(0,n-get_players(n)+2)})

((1, 2, 4), (2, 3, 5), (3, 4, 6), (0, 4, 5), (1, 5, 6), (0, 2, 6), (0, 1, 3))
{0: True, 1: True, 2: True, 3: True, 4: True, 5: True}


Now let's modify the tournament such that it no longer works, removing the last value from the set.  As a result, (0,3) and (1,3) no longer have any items in the "winners" list that contain both those values, and as a result, 0 and 1 should fail but the others should remain unaffected:

In [5]:
modified_winners = tuple(list(winners[:-1])+[winners[-1][:-1]])
print(modified_winners)
print({i:has_full_coverage(modified_winners,n,i) for i in range(0,n-get_players(n)+2)})

((1, 2, 4), (2, 3, 5), (3, 4, 6), (0, 4, 5), (1, 5, 6), (0, 2, 6), (0, 1))
{0: False, 1: False, 2: True, 3: True, 4: True, 5: True}


So we have confidence that our coverage function is working as expected, so we can parallelize it using a function:

In [6]:
def check(d2,dice): return tuple([d1[0] for d1 in dice if get_win_bias(d1,d2)>0])
def get_winners_parallel(dice): return tuple(Parallel(n_jobs=20,verbose=2)(delayed(check)(d2,dice) for d2 in dice))
def check_comb_parallel(dice,n):
    print('generating winner table')
    winners = get_winners_parallel(dice)
    print('checking combinations of solutions')
    return all(Parallel(n_jobs=20,verbose=2)(delayed(has_full_coverage)(winners,n,i) for i in range(0,n-get_players(n)+2)))

And the Li and Marshall dice for a 5-player solutions work:

In [7]:
n=67
check_comb_parallel(li_dice(n),n)

generating winner table


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done  62 out of  67 | elapsed:    0.5s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  67 out of  67 | elapsed:    0.5s finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.1s


checking combinations of solutions


[Parallel(n_jobs=20)]: Done  58 out of  64 | elapsed:    0.3s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  64 out of  64 | elapsed:    0.3s finished


True

In [8]:
n=71
check_comb_parallel(marshall_dice(n),n)

[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=20)]: Done  68 out of  71 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  71 out of  71 | elapsed:    0.0s finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


generating winner table
checking combinations of solutions


[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    0.2s
[Parallel(n_jobs=20)]: Done  64 out of  68 | elapsed:    0.3s remaining:    0.0s
[Parallel(n_jobs=20)]: Done  68 out of  68 | elapsed:    0.4s finished


True

It should be noted that once the winner table is generated, whichever has the smallest number of dice will simply run faster.

Now for checking the six-player solution, the Marshall dice have the restriction that n = 7 (mod 8), of which the possible values for n>=331 required for a 6-player are given:

In [9]:
[i for i in range(3,400,8) if i>=331 and __import__('sympy').isprime(i)]

[331, 347, 379]

Let's check the Li dice first though:

In [10]:
n=331
check_comb_parallel(li_dice(n),n)

generating winner table


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:    1.8s
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed:  1.0min
[Parallel(n_jobs=20)]: Done 331 out of 331 | elapsed:  2.8min finished
[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.


checking combinations of solutions


KeyboardInterrupt: 

So there we have it.  The Dice given by Li represent a working solution to the six-player intransitive dice problem.  We can also attempt this with the Marshall dice form for 6 players:

In [ ]:
n=347
check_comb_parallel(marshall_dice(n),n)

In [ ]:
print(tabulate(marshall_dice(347)))

I won't print out the Li dice because they are simply too large, but the Marshall dice that solve the 6-player problem in the fewest faces are given above.